# ⚡ Notebook 4 — GPU Intel con OpenVINO su WSL2

In questo notebook imparerai a:
1. **Verificare** che la GPU Intel sia accessibile da WSL2
2. **Caricare** il modello su GPU invece che su CPU
3. **Confrontare** la velocità (token/s) tra CPU e GPU
4. Usare la modalità **AUTO** che sceglie il dispositivo migliore

> **Nota WSL2**: la GPU Intel funziona tramite `/dev/dxg`, un device virtuale
> creato da WSL2 come bridge verso i driver DirectX di Windows.
> La NPU Intel **non è supportata** in WSL2 (limite del kernel WSL).

---
## 🔍 Passo 1: Verifica il device /dev/dxg

In [ ]:
import os
from pathlib import Path

dxg = Path("/dev/dxg")

if dxg.exists():
    print("✅ /dev/dxg trovato — la GPU Intel è accessibile da WSL2")
else:
    print("❌ /dev/dxg NON trovato.")
    print()
    print("Possibili cause:")
    print("  1. Il container non è stato avviato con --profile gpu")
    print("     → Riavvia con: docker compose --profile gpu up -d")
    print()
    print("  2. Il driver Intel su Windows non è aggiornato")
    print("     → Installa driver >= 30.0.100.9955 da:")
    print("       https://www.intel.com/content/www/us/en/download/19344/")
    print()
    print("  3. WSL2 non è aggiornato")
    print("     → Da PowerShell: wsl --update")

---
## 🖥️ Passo 2: Dispositivi rilevati da OpenVINO

In [ ]:
import openvino as ov
import openvino_genai as ov_genai
import time

core = ov.Core()

print("Dispositivi rilevati da OpenVINO:")
print("=" * 55)
for device in core.available_devices:
    try:
        nome = core.get_property(device, "FULL_DEVICE_NAME")
        print(f"  {device:6s}  →  {nome}")
    except Exception:
        print(f"  {device:6s}  →  (info non disponibili)")

print()
print("  AUTO   →  OpenVINO sceglie automaticamente")
print()

# Dispositivo da usare (letto dalla variabile d'ambiente del container)
DEVICE = os.environ.get("OV_DEVICE", "CPU")
print(f"Dispositivo configurato nel container: {DEVICE}")

if DEVICE not in core.available_devices and DEVICE != "AUTO":
    print(f"⚠️  '{DEVICE}' non disponibile, uso CPU")
    DEVICE = "CPU"
else:
    print(f"✅ '{DEVICE}' disponibile")

---
## 🔧 Passo 3: Carica il Modello

> **Nota sulla GPU**: la prima volta OpenVINO compila i kernel OpenCL per il tuo
> specifico modello di GPU (~30-60s). Il risultato viene salvato nella cache
> `/workspace/models/.cache_gpu/` — le volte successive il caricamento è molto più veloce.

In [ ]:
MODEL_DIR = "/workspace/models/tinyllama-chat-ov"

config = {}
if DEVICE == "GPU":
    config["CACHE_DIR"] = "/workspace/models/.cache_gpu"

print(f"Caricamento su {DEVICE}...")
if DEVICE == "GPU":
    print("(prima esecuzione: ~30-60s per compilazione kernel OpenCL)")

t0 = time.time()
pipe = ov_genai.LLMPipeline(MODEL_DIR, DEVICE, **config)
print(f"✅ Caricato in {time.time()-t0:.1f}s")

---
## 🏎️ Passo 4: Benchmark — Token al Secondo

In [ ]:
PROMPT_TEST = (
    "<|system|>\nRispondi in italiano.</s>\n"
    "<|user|>\nSpiega in 5 frasi cos'è una rete neurale artificiale.</s>\n"
    "<|assistant|>\n"
)

def misura_velocita(pipeline, label, n_runs=3):
    cfg = ov_genai.GenerationConfig()
    cfg.max_new_tokens = 80
    cfg.do_sample = False
    
    risultati = []
    for i in range(n_runs):
        tokens = []
        t0 = time.time()
        pipeline.generate(PROMPT_TEST, cfg, lambda t: (tokens.append(t), False)[1])
        elapsed = time.time() - t0
        tok_s = len(tokens) / elapsed
        risultati.append(tok_s)
        print(f"  Run {i+1}: {len(tokens):3d} token · {elapsed:.2f}s · {tok_s:.1f} tok/s")
    
    media = sum(risultati) / len(risultati)
    print(f"  {'─'*40}")
    print(f"  Media {label}: {media:.1f} tok/s")
    return media

print(f"\n⏱️  Benchmark su {DEVICE}:")
tok_s_corrente = misura_velocita(pipe, DEVICE)

---
## 🔄 Passo 5: Confronto CPU vs GPU

Carichiamo anche la CPU per confrontare direttamente.

In [ ]:
risultati_benchmark = {DEVICE: tok_s_corrente}

# Carica il dispositivo alternativo per confronto
altro = "CPU" if DEVICE == "GPU" else "GPU"

if altro in core.available_devices:
    print(f"\nCaricamento {altro} per confronto...")
    cfg_altro = {"CACHE_DIR": "/workspace/models/.cache_gpu"} if altro == "GPU" else {}
    pipe_altro = ov_genai.LLMPipeline(MODEL_DIR, altro, **cfg_altro)
    print(f"\n⏱️  Benchmark su {altro}:")
    risultati_benchmark[altro] = misura_velocita(pipe_altro, altro)
else:
    print(f"'{altro}' non disponibile, salto il confronto.")

# Grafico ASCII
print("\n" + "═" * 50)
print("📊 RISULTATI FINALI")
print("═" * 50)
max_v = max(risultati_benchmark.values())
for dev, v in sorted(risultati_benchmark.items(), key=lambda x: -x[1]):
    barra = "█" * int(v / max_v * 35)
    print(f"  {dev:4s}  {barra:<35s}  {v:.1f} tok/s")

if "CPU" in risultati_benchmark and "GPU" in risultati_benchmark:
    speedup = risultati_benchmark["GPU"] / risultati_benchmark["CPU"]
    print(f"\n  Speedup GPU/CPU: {speedup:.1f}x")

---
## 🤖 Passo 6: Genera una Risposta con il Dispositivo Scelto

In [ ]:
# ✏️ Scrivi qui la tua domanda!
domanda = "Qual è la differenza tra CPU e GPU nell'intelligenza artificiale?"

prompt = (
    f"<|system|>\nSei un assistente didattico. Rispondi in italiano.</s>\n"
    f"<|user|>\n{domanda}</s>\n"
    f"<|assistant|>\n"
)

cfg = ov_genai.GenerationConfig()
cfg.max_new_tokens = 200
cfg.temperature = 0.7
cfg.do_sample = True

print(f"🤖 Risposta ({DEVICE}):\n")
print("─" * 50)
tokens = []
t0 = time.time()
pipe.generate(prompt, cfg, lambda t: (print(t, end="", flush=True), tokens.append(t), False)[2])
print(f"\n─ {len(tokens)} token · {time.time()-t0:.1f}s · {len(tokens)/(time.time()-t0):.1f} tok/s")